# Sanitize searchable PDFs using the ASE analyzer

Detect English/US PII in native-text, scanned, and mixed PDFs and replace detected regions with their entity labels: `<EMAIL_ADDRESS>`, `<US_SSN>`, `<PHONE_NUMBER>`, `<PERSON>`, `<LOCATION>`, `<CREDIT_CARD>`, and other types returned by your analyzer.

This notebook uses your existing authenticated [ASE connection](presidio_ase_notebook.ipynb). No local Presidio models or anonymizer service are needed. OCR and PDF processing run on the kernel host; extracted text is sent to your ASE analyzer. It does not change Azure resources or permissions.

Each page is rendered and analyzed using both whole-page OCR and native text. By default, `PDF_LAYOUT = "reflow"` creates a **new searchable, text-only A4 PDF** with complete entity labels, line wrapping, and extra pages as needed. Labels no longer have to fit the original PII rectangles. Original page images, text layers, and hidden PDF objects are not copied. Only OCR words completely outside detected regions and exact replacement labels enter the new document.

**Detection is not a guarantee of PII-free output.** Review results privately. Do not display original text, matched values, tokens, service error bodies, or real document images. Start with synthetic input and clear outputs before sharing.


## Kernel prerequisites

Copy this notebook **and `ase_document_tools.py`** to the kernel host. Use Python 3.10 through 3.14 and install dependencies in the selected uv-managed environment (Windows terminal example):
```powershell
uv venv .venv-ase
uv pip install --python .venv-ase\Scripts\python.exe requests azure-identity ipykernel Pillow "PyMuPDF>=1.24,<2"
```
**PyMuPDF uses AGPL/commercial licensing; confirm the appropriate license before installation.** Separately install Tesseract OCR and English language data on the kernel host; a Python package does not install the OCR engine. Select this environment as the notebook kernel.

The host must reach the analyzer over private DNS/HTTPS and expose the managed identity used by your working connection notebook. Keep tenant/caller restrictions and TLS verification enabled. Configure `REQUESTS_CA_BUNDLE` for a private CA if needed. Reuse the registered analyzer Application ID URI (without `/.default`) and calling identity client ID. Do not guess resource identifiers or broaden permissions to bypass errors.


In [ ]:
import os
import sys
from pathlib import Path

TOOLS_DIRECTORY = Path(os.environ.get("ASE_SAMPLE_DIRECTORY", str(Path.cwd())))
ANALYZER_BASE_URL = os.environ.get("ANALYZER_BASE_URL", "")
ANALYZER_API_RESOURCE = os.environ.get("ANALYZER_API_RESOURCE", "")
MANAGED_IDENTITY_CLIENT_ID = os.environ.get("MANAGED_IDENTITY_CLIENT_ID", "")
INPUT_DOCUMENT = Path(
    os.environ.get("INPUT_DOCUMENT", str(Path("sample_data") / "sample.pdf"))
)
OUTPUT_DOCUMENT = Path(
    os.environ.get(
        "OUTPUT_DOCUMENT",
        str(INPUT_DOCUMENT.with_name(INPUT_DOCUMENT.stem + "_sanitized.pdf")),
    )
)
LANGUAGE = "en"
OCR_LANGUAGE = "eng"
TESSERACT_COMMAND = os.environ.get("TESSERACT_COMMAND", "tesseract")
RENDER_DPI = 200
OCR_MIN_CONFIDENCE = 50
MIN_LABEL_FONTSIZE = 6
PDF_LAYOUT = "reflow"

if not (TOOLS_DIRECTORY / "ase_document_tools.py").is_file():
    raise FileNotFoundError(
        "Copy ase_document_tools.py to the kernel host and set ASE_SAMPLE_DIRECTORY."
    )
sys.path.insert(0, str(TOOLS_DIRECTORY.resolve()))

In [ ]:
from ase_document_tools import ASEAnalyzer, sanitize_pdf

## Coverage and layout policy

- All returned English analyzer entities are processed, not only email. Actual coverage depends on the deployed recognizers and NLP configuration. US SSNs and other identifiers may need context/checksum-valid input. `LOCATION` is **not comprehensive street-address detection**; this notebook adds no recognizers.
- Whole-page OCR includes embedded images and visible annotations/forms. Native character geometry supplements OCR. Duplicate/overlapping regions are merged with all associated types in alphabetical order; multiline matches receive a label on each affected line. Counts may overlap across native text and OCR.
- In `reflow` mode, output uses fresh A4 pages rather than the original layout. Text and complete labels wrap at spaces and paginate without clipping. A label is never split internally. Each source page starts a new output page; output page counts can exceed input page counts. Fonts normally use 12 points, reducing only when necessary to fit a long token, never below `MIN_LABEL_FONTSIZE`. An exceptionally long word or an excessive minimum font size can still refuse output.
- Intersecting OCR words are excluded in full, even for partial-word matches. Original OCR line order is retained where possible; columns/tables may read incorrectly and graphics are omitted. Native detections without corresponding OCR words receive complete labels at the end of that source page's text. Missing non-PII OCR text cannot be recovered by reflow.
- Optional `PDF_LAYOUT = "preserve"` retains sanitized page images, displayed dimensions, and a filtered invisible OCR layer. This original mode still requires full labels to fit their removed regions at the minimum font size and refuses output when they cannot. Use `reflow`, not a smaller minimum, to resolve label-fit errors.
- Searchable text contains OCR-recognized non-PII words and exact labels, not original PDF text. Recognition/reading order and word positions may differ. The sample font supports Latin-1 text; unsupported characters refuse output rather than silently corrupting search text.
- Original attachments, metadata, scripts, links, forms, annotations, and hidden text are not copied. Original accessibility tags, editability, and vector quality are lost. Visible form/annotation content is flattened into pixels before analysis.


## Optional synthetic smoke test

This runs against your actual ASE analyzer and Tesseract using a temporary PDF containing only invented test values. It verifies exact labels, retained searchable public text, and absence of the test values in extracted output. It does not certify real-document detection or validate every entity type. Set `RUN_SYNTHETIC_TEST = True` to run it; failures require reviewing the deployed recognizer configuration or OCR, not suppressing the checks.


In [ ]:
RUN_SYNTHETIC_TEST = False

if RUN_SYNTHETIC_TEST:
    import tempfile

    import pymupdf

    with tempfile.TemporaryDirectory() as folder:
        source = Path(folder) / "synthetic.pdf"
        target = Path(folder) / "synthetic_sanitized.pdf"
        with pymupdf.open() as pdf:
            page = pdf.new_page(width=700, height=400)
            page.insert_text((40, 50), "Public heading", fontsize=18)
            page.insert_text((40, 120), "Email: alex@example.com", fontsize=20)
            page.insert_text(
                (40, 200), "Social security number: 321-54-6789", fontsize=20
            )
            pdf.save(source)
        with ASEAnalyzer(
            ANALYZER_BASE_URL,
            ANALYZER_API_RESOURCE,
            MANAGED_IDENTITY_CLIENT_ID,
            LANGUAGE,
        ) as analyzer:
            sanitize_pdf(
                source,
                target,
                analyzer.analyze,
                dpi=RENDER_DPI,
                ocr_language=OCR_LANGUAGE,
                tesseract=TESSERACT_COMMAND,
                min_confidence=OCR_MIN_CONFIDENCE,
                min_label_fontsize=MIN_LABEL_FONTSIZE,
                layout=PDF_LAYOUT,
            )
        with pymupdf.open(target) as pdf:
            sanitized_text = pdf[0].get_text()
            if not all(
                label in sanitized_text for label in ("<EMAIL_ADDRESS>", "<US_SSN>")
            ):
                raise RuntimeError(
                    "Synthetic entity-label check failed; "
                    "review analyzer/OCR configuration."
                )
            if any(
                value in sanitized_text for value in ("alex@example.com", "321-54-6789")
            ):
                raise RuntimeError("Synthetic searchable-text removal check failed.")
            if "Public" not in sanitized_text:
                raise RuntimeError("Synthetic non-PII searchability check failed.")
        print("Synthetic email/SSN label and searchable-text checks passed.")

## Sanitize your PDF

Set paths on the kernel host and choose a **new output filename in an existing directory**. Reruns do not overwrite input or output. The sample limits input to 100 MiB, 200 pages, 25 million rendered pixels per page, and 100,000 characters per analyzer request. Rendering requires at least 144 DPI. `LANGUAGE` uses the analyzer's ISO code (`en`); `OCR_LANGUAGE` uses installed Tesseract language data (`eng`).

Missing OCR/language data, low-confidence OCR, nonuniform pages with no recognized text, invalid detections/geometry, unrepresentable text, label-fit failures, encrypted PDFs, and service errors refuse output. Original rendered pages and OCR text are not saved separately. Publication requires same-directory filesystem hard-link support and never replaces an existing file.


In [ ]:
with ASEAnalyzer(
    ANALYZER_BASE_URL, ANALYZER_API_RESOURCE, MANAGED_IDENTITY_CLIENT_ID, LANGUAGE
) as analyzer:
    summary = sanitize_pdf(
        INPUT_DOCUMENT,
        OUTPUT_DOCUMENT,
        analyzer.analyze,
        dpi=RENDER_DPI,
        ocr_language=OCR_LANGUAGE,
        tesseract=TESSERACT_COMMAND,
        min_confidence=OCR_MIN_CONFIDENCE,
        min_label_fontsize=MIN_LABEL_FONTSIZE,
        layout=PDF_LAYOUT,
    )
print(
    f"Saved searchable sanitized PDF: {summary['pages']} pages; "
    f"{summary['text_detections']} native-text and "
    f"{summary['ocr_detections']} OCR detections."
)
print("Replacement-label counts (not unique PII counts):", summary["label_types"])

## Private review before use

Inspect rendered output and search/copy text privately. Verify labels replace detected content and surrounding content remains usable. A success message or absence of known values in searchable text does not prove pixel-level removal or complete detection. Handwriting, unusual fonts/orientations, photos, complex layouts, unrecognized text, and missing recognizers can leave PII undetected even when OCR confidence is high. Protect both input and output accordingly.

For authentication failures, reuse the working ASE resource/identity, issuer/token version, caller restrictions, and private-network routing. Keep TLS verification enabled. Do not print raw HTTP bodies or loosen access controls to troubleshoot.
